In [15]:

from pyspark.sql import SparkSession
import pyspark.sql.functions as F

spark = (
    SparkSession.builder
    .appName('big-data-management-assignment-1')
    .enableHiveSupport()   # persist tables to local Hive metastore
    .getOrCreate()
)

print(spark.version)

4.1.0


In [16]:
# Step 1: Incremental Ingestion - Schema Definition
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, LongType, DoubleType, TimestampType
import json
import os
from pathlib import Path
from datetime import datetime

# Define explicit schema for NYC Yellow Taxi Trip Records
# Based on TLC data dictionary: https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page
# Note: Parquet files store integer columns as INT64, so we use LongType instead of IntegerType
yellow_taxi_schema = StructType([
    StructField("VendorID", LongType(), True),
    StructField("tpep_pickup_datetime", TimestampType(), True),
    StructField("tpep_dropoff_datetime", TimestampType(), True),
    StructField("passenger_count", LongType(), True),
    StructField("trip_distance", DoubleType(), True),
    StructField("RatecodeID", LongType(), True),
    StructField("store_and_fwd_flag", StringType(), True),
    StructField("PULocationID", LongType(), True),
    StructField("DOLocationID", LongType(), True),
    StructField("payment_type", LongType(), True),
    StructField("fare_amount", DoubleType(), True),
    StructField("extra", DoubleType(), True),
    StructField("mta_tax", DoubleType(), True),
    StructField("tip_amount", DoubleType(), True),
    StructField("tolls_amount", DoubleType(), True),
    StructField("improvement_surcharge", DoubleType(), True),
    StructField("total_amount", DoubleType(), True),
    StructField("congestion_surcharge", DoubleType(), True),
    StructField("airport_fee", DoubleType(), True)
])

print(f"Schema defined with {len(yellow_taxi_schema.fields)} fields")

Schema defined with 19 fields


In [17]:
# Manifest Handling Functions

MANIFEST_PATH = "state/manifest.json"

def load_manifest():
    """Load the manifest of already processed files."""
    if os.path.exists(MANIFEST_PATH):
        with open(MANIFEST_PATH, 'r') as f:
            return json.load(f)
    return {"processed_files": [], "last_updated": None}

def save_manifest(manifest):
    """Save the manifest to disk."""
    os.makedirs(os.path.dirname(MANIFEST_PATH), exist_ok=True)
    
    # Add timestamp
    manifest["last_updated"] = datetime.now().isoformat()
    
    # Write to file with pretty formatting
    with open(MANIFEST_PATH, 'w') as f:
        json.dump(manifest, indent=2, fp=f)
    print(f"Manifest saved: {len(manifest['processed_files'])} files tracked")

def get_new_files(input_dir, manifest):
    """Identify files in input_dir that haven't been processed yet."""
    processed_files = set(manifest.get("processed_files", []))
    
    # List all parquet files in input directory
    input_path = Path(input_dir)
    all_files = [str(f.relative_to(input_path.parent)) for f in input_path.glob("yellow_tripdata_*.parquet")]
    
    # Filter to only new files
    new_files = [f for f in all_files if f not in processed_files]
    
    return sorted(new_files)

# Test manifest functions
manifest = load_manifest()
print(f"Current manifest: {len(manifest.get('processed_files', []))} files already processed")

Current manifest: 0 files already processed


In [18]:
# Incremental File Reading Function

def read_incremental_data(input_dir, manifest, schema):
    """
    Read only new parquet files that haven't been processed yet.
    
    Args:
        input_dir: Directory containing input parquet files
        manifest: Dict containing already processed files
        schema: Explicit StructType schema for reading
    
    Returns:
        tuple: (DataFrame with new data, updated manifest dict)
               Returns (None, manifest) if no new files found
    """
    new_files = get_new_files(input_dir, manifest)
    
    if not new_files:
        print("No new files to process")
        return None, manifest
    
    print(f"Found {len(new_files)} new file(s) to process:")
    for f in new_files:
        print(f"   • {f}")
    
    # Read all new files with explicit schema
    file_paths = [os.path.join(input_dir, f.split('/')[-1]) for f in new_files]
    
    # Configure Spark for optimized reads
    # maxPartitionBytes controls split size - 128MB 
    spark.conf.set("spark.sql.files.maxPartitionBytes", "128MB")
    
    # Read parquet files with explicit schema
    df = spark.read.schema(schema).parquet(*file_paths)
    
    # Add metadata columns for lineage tracking
    # - source_file: which parquet file this row came from
    # - ingested_at: when this batch was processed
    df = (df
          .withColumn("source_file", F.input_file_name())
          .withColumn("ingested_at", F.lit(datetime.now().isoformat()))
    )
    
    # Get row counts per file
    row_count = df.count()
    print(f"Read {row_count:,} rows from {len(new_files)} file(s)")
    
    # Update manifest
    manifest["processed_files"].extend(new_files)
    
    return df, manifest


In [19]:
# Execute Incremental Ingestion

# Configuration
INPUT_DIR = "data/input"

# Load manifest
manifest = load_manifest()

# Read only new files
raw_df, manifest = read_incremental_data(INPUT_DIR, manifest, yellow_taxi_schema)

if raw_df is not None:
    # Show sample of data
    print("\nSample of ingested data:")
    raw_df.select(
        "tpep_pickup_datetime", 
        "PULocationID", 
        "DOLocationID", 
        "passenger_count", 
        "trip_distance",
        "source_file",
        "ingested_at"
    ).show(5, truncate=80)
    
    # Show data summary
    print("\nData Summary:")
    print(f"   Total rows: {raw_df.count():,}")
    print(f"   Columns: {len(raw_df.columns)}")
    print(f"   Partitions: {raw_df.rdd.getNumPartitions()}")
    
    # Save updated manifest
    save_manifest(manifest)
    
    print("\nStep 1 Complete - Data ready for transformation")
else:
    print("\nNo new data to process")

Found 2 new file(s) to process:
   • input/yellow_tripdata_2025-01.parquet
   • input/yellow_tripdata_2025-02.parquet
Read 7,052,769 rows from 2 file(s)

Sample of ingested data:
+--------------------+------------+------------+---------------+-------------+-------------------------------------------------------------------+--------------------------+
|tpep_pickup_datetime|PULocationID|DOLocationID|passenger_count|trip_distance|                                                        source_file|               ingested_at|
+--------------------+------------+------------+---------------+-------------+-------------------------------------------------------------------+--------------------------+
| 2025-01-01 00:18:38|         229|         237|              1|          1.6|file:///home/jovyan/work/data/input/yellow_tripdata_2025-01.parquet|2026-03-01T22:07:21.116924|
| 2025-01-01 00:32:40|         236|         237|              1|          0.5|file:///home/jovyan/work/data/input/yellow_trip

In [20]:
# Inspect Schema and Data Quality

if raw_df is not None:
    
    # Show schema to verify types
    print("\nSchema (verify types are correct):")
    raw_df.printSchema()
    
    # Check for nulls in critical columns
    print("\nNull count in critical columns:")
    raw_df.select([
        F.sum(F.when(F.col("tpep_pickup_datetime").isNull(), 1).otherwise(0)).alias("null_pickup_dt"),
        F.sum(F.when(F.col("PULocationID").isNull(), 1).otherwise(0)).alias("null_pickup_loc"),
        F.sum(F.when(F.col("DOLocationID").isNull(), 1).otherwise(0)).alias("null_dropoff_loc"),
        F.sum(F.when(F.col("trip_distance").isNull(), 1).otherwise(0)).alias("null_distance")
    ]).show()
    
    # Basic statistics on numeric columns
    print("\nBasic statistics:")
    raw_df.select("trip_distance", "passenger_count", "total_amount").describe().show()
    
else:
    print("No data loaded - skipping quality checks")


Schema (verify types are correct):
root
 |-- VendorID: long (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)
 |-- source_file: string (nullable = false)
 |-- ingested_at: string (nullable = false)


Null count in cri

In [21]:
# ============================================================
# Optional: Test Incremental Behavior (Idempotency Check)
# ============================================================
# This cell verifies that re-running won't duplicate data.
# Expected: After first run, this shows 0 new files.
# ============================================================

# Load current manifest state
test_manifest = load_manifest()

print(f"\nFiles currently in manifest: {len(test_manifest.get('processed_files', []))}")
if test_manifest.get('processed_files'):
    for idx, file in enumerate(test_manifest.get('processed_files', []), 1):
        print(f"   {idx}.  {file}")
else:
    print("   (empty - no files processed yet)")

# Check for new files
test_new = get_new_files(INPUT_DIR, test_manifest)

print(f"\nNew files detected: {len(test_new)}")
if test_new:
    print(" Files waiting to be processed:")
    for f in test_new:
        print(f"      • {f}")
    print("\n  Run the execution cell above to process these files")
else:
    print("  No new files found")
    print("  Incremental logic working correctly - no duplicates!")
    print("\n To test incremental behavior:")
    print("      1. Add a new yellow_tripdata_YYYY-MM.parquet to data/input/")
    print("      2. Re-run the execution cell above")
    print("      3. Only the new file will be processed")



Files currently in manifest: 2
   1.  input/yellow_tripdata_2025-01.parquet
   2.  input/yellow_tripdata_2025-02.parquet

New files detected: 0
  No new files found
  Incremental logic working correctly - no duplicates!

 To test incremental behavior:
      1. Add a new yellow_tripdata_YYYY-MM.parquet to data/input/
      2. Re-run the execution cell above
      3. Only the new file will be processed
